In [1]:
import winsound
from tqdm.auto import tqdm
import sympy as sp
from sympy.core.function import AppliedUndef
from sympy.physics.wigner import clebsch_gordan
from sympy.physics.quantum.spin import Rotation as rot
from sympy import pi
from random import randint
import psutil
import dill
import os

sp.init_printing(use_unicode=True, wrap_line=True, num_columns=160)

import time
start_time = time.time()

Jmax = 2
Lmax = 2

In [2]:
# --- Definizione simboli ---
J, LB, LR, L1, L2, JX, lam, lam1, lam2, eta = sp.symbols('J LB LR L1 L2 JX lam lam1 lam2 eta', integer=True)
lambda1, lambda2 = sp.symbols('lambda1 lambda2', real=True)
theta1, theta2, thetal, phi, chi = sp.symbols('theta1 theta2 thetal phi chi', real=True)
eta_par = sp.symbols('eta_par', integer=True)
Res = sp.symbols('Res')

# --- Helicity amplitudes e altri coefficienti simbolici ---
H = sp.Function('H', complex=True)(eta, J, lam, LB, Res)     # Helicity amplitude per B -> K_J
Hbar = sp.Function('Hbar', complex=True)(eta, J, lam, LB, Res)     # Helicity amplitude per B -> K*_J
LHA = sp.Function('LHA', real=True)(eta_par, lam1, lam2)  # Leptonic Helicity amplitude
G = sp.Function('G', complex=True)(eta, J, lam, LB, Res)  # Helicity Amplitude for K_J -> R P1
NR = sp.Function('NR', complex=True)(eta)

# --- Breit-Wigner propagatore ---
Prop = sp.Function('R')(Res)

# --- Blatt-Weisskopf factor ---
L, rBW = sp.symbols('L rBW', positive=True, real=True)
rB, r3, r2 = sp.symbols('rB r3 r2', positive=True, real=True)
q = sp.symbols('q', real=True)
qR = sp.symbols('qR', real=True)

# funzione simbolica generica
BWfunc = sp.Function('BW', real=True)

# istanza base (con argomenti simbolici)
BW = BWfunc(L, rBW, qR)

atot = []
a_NR = sp.symbols('a_NR', complex=True)
atot.append(a_NR)  # contributo non-risonante

In [3]:
### Cella per costruire l'ampiezza A3 ###

# Lista delle risonanze come tuple (nome, spin)
#(R, J, LR, JX, a_i), dove LR è il mom. ang. orb. tra P1 e (P2P3), JX è il mom. ang. orb. tra P2 e P3 (uguale allo spin JX).
list_reso3 = ['K(1460)', 'K1(1650)', 'K2(1770)', 'K2(1820)']
resonances3 = {
    ('K(1460)', 0, 0, 0): sp.symbols(f'a1', complex=True), #1
    ('K1(1650)', 1, 0, 1): sp.symbols(f'a2', complex=True), #2
    ('K1(1650)', 1, 1, 0): sp.symbols(f'a3', complex=True), #3
    ('K2(1820)', 2, 0, 2): sp.symbols(f'a4', complex=True), #4
    ('K2(1820)', 2, 2, 0): sp.symbols(f'a5', complex=True), #5
}


a3 = []
for a in resonances3.values():
    a3.append(a)
atot.extend(a3)

# --- Costruzione dell'ampiezza A3 ---
A3 = 0  # Inizializza l'ampiezza

for J_val in range(0, Jmax + 1):
    for LB_val in range(abs(J_val-1), abs(J_val-1) + 1):
        print(f"\n\nCalcolando contributi per J = {J_val}, LB = {LB_val}")
        # Somma su tutte le risonanze R con spin = J_val
        for idx_res, keys in enumerate(resonances3.items()):
            R_name, spin, LR, JX = keys[0]
            coeff_a = keys[1]
            if spin == J_val:
                R_symbol = sp.Symbol(R_name)
                for JX_val in [JX]:
                    for lamx_val in range(-JX_val, JX_val + 1):
                        for LR_val in range(abs(J_val - JX_val), abs(J_val - JX_val) + 1):
                            for lam_val in range(-min(1,J_val), min(1,J_val) + 1):
                                H_term = H.subs(J, J_val).subs(LB, LB_val).subs(lam, lam_val).subs(Res, R_symbol)
                                # usa il coefficiente corrispondente dalla lista a3
                                term = (
                                    coeff_a * H_term * clebsch_gordan(J_val, 1, LB_val, lam_val, -lam_val, 0) *
                                    LHA.subs(eta_par, eta).subs(lam1, lambda1).subs(lam2, lambda2) *
                                    sp.conjugate(rot.D(1, lam_val, lambda1 - lambda2, 0, thetal, 0)) * 
                                    sp.sqrt(sp.Rational((2*LR_val + 1),(2*J_val + 1))) * 
                                    sp.sqrt(sp.Rational((2*J_val + 1),4)/pi) * sp.sqrt(sp.Rational((2*JX_val + 1),4)/pi) * 
                                    clebsch_gordan(LR_val, JX_val, J_val, lam_val - lamx_val, lamx_val, lam_val) *
                                    sp.conjugate(rot.D(J_val, lam_val, lamx_val, phi, theta1, 0)) *
                                    sp.conjugate(rot.D(JX_val, lamx_val, 0, chi, theta2, 0)) *
                                    Prop.subs(Res, R_symbol) *
                                    BW.subs(L, LB_val).subs(rBW, rB).subs(qR, sp.Symbol('B')) * BW.subs(L, J_val).subs(rBW, r3).subs(qR, R_symbol)
                                    )
                                A3 += term
                                print(f"\nR = {R_name}, (spin = {spin})")
                                print(f"Processando {R_name} (spin {J_val}) lambda = {lam_val}, LB = {LB_val}")
                                print(f"JX = {JX_val}, lambdaX = {lamx_val}, LR = {LR_val}")
                                print(f"term = {term}")

# Mostra l'ampiezza totale
print("\n" + "="*50)
print("Ampiezza A3 totale:")
if A3 == 0:
    print("L'ampiezza A3 è nulla (nessuna risonanza selezionata).")
else:
    print(f"Termini nell'ampiezza A3: {len(A3.as_ordered_terms())}")
print("="*50)

# Mostra l'ampiezza
sp.pprint(A3)



Calcolando contributi per J = 0, LB = 1

R = K(1460), (spin = 0)
Processando K(1460) (spin 0) lambda = 0, LB = 1
JX = 0, lambdaX = 0, LR = 0
term = a1*BW(0, r3, K(1460))*BW(1, rB, B)*H(eta, 0, 0, 1, K(1460))*LHA(eta, lambda1, lambda2)*R(K(1460))*conjugate(WignerD(0, 0, 0, chi, theta2, 0))*conjugate(WignerD(0, 0, 0, phi, theta1, 0))*conjugate(WignerD(1, 0, lambda1 - lambda2, 0, thetal, 0))/(4*pi)


Calcolando contributi per J = 1, LB = 0

R = K1(1650), (spin = 1)
Processando K1(1650) (spin 1) lambda = -1, LB = 0
JX = 1, lambdaX = -1, LR = 0
term = a2*BW(0, rB, B)*BW(1, r3, K1(1650))*H(eta, 1, -1, 0, K1(1650))*LHA(eta, lambda1, lambda2)*R(K1(1650))*conjugate(WignerD(1, -1, -1, phi, theta1, 0))*conjugate(WignerD(1, -1, 0, chi, theta2, 0))*conjugate(WignerD(1, -1, lambda1 - lambda2, 0, thetal, 0))/(4*pi)

R = K1(1650), (spin = 1)
Processando K1(1650) (spin 1) lambda = 0, LB = 0
JX = 1, lambdaX = -1, LR = 0
term = 0

R = K1(1650), (spin = 1)
Processando K1(1650) (spin 1) lambda = 1, LB = 

In [6]:
### Cella per costruire l'ampiezza AR ###

# Lista delle risonanze come tuple (nome, spin)
list_resoR = ['rho(770)', 'omega(782)', 'phi(1020)', 'f2(1270)', 'f0(1500)', 'f2(1525)',
              'K0*(700)', 'K*(892)', 'K*(1410)', 'K0*(1430)', 'K2*(1430)', 'K*(1680)', 'K0*(1950)']
resonancesR = {
    #(R, spin, L1, L2), dove L1 è il mom. ang. orb. tra P1 e R, R2 è uguale allo spin di R.
    ('rho(770)', 1, 0, 1): sp.symbols(f'a6', complex=True), #6
    ('omega(782)', 1, 0, 1): sp.symbols(f'a7', complex=True), #7
    ('phi(1020)', 1, 0, 1): sp.symbols(f'a8', complex=True), #8
    ('f2(1270)', 2, 0, 2): sp.symbols(f'a9', complex=True), #9
    ('f0(1500)', 0, 0, 0): sp.symbols(f'a10', complex=True), #10
    ('K*(892)', 1, 0, 1): sp.symbols(f'a11', complex=True), #11
    ('K*(1410)', 1, 0, 1): sp.symbols(f'a12', complex=True), #12
    ('K0*(1430)', 0, 0, 0): sp.symbols(f'a13', complex=True), #13
    ('K2*(1430)', 2, 0, 2): sp.symbols(f'a14', complex=True), #14
    ('K*(1680)', 1, 0, 1): sp.symbols(f'a15', complex=True), #15
    ('K0*(1950)', 0, 0, 0): sp.symbols(f'a16', complex=True), #16
}

aR = []
for a in resonancesR.values():
    aR.append(a)
atot.extend(aR)

# --- Costruzione dell'ampiezza AR ---
AR = 0
for J_val in range(0, Jmax + 1):
    for LB_val in range(abs(J_val-1), abs(J_val-1) + 1):
        print(f"\n\nCalcolando contributi per J = {J_val}, LB = {LB_val}")
        for idx_res, keys in enumerate(resonancesR.items()):
            R_name, spin, L1, L2 = keys[0]
            coeff_a = keys[1]
            if spin == J_val:
                R_symbol = sp.Symbol(R_name)    
                for lam_val in range(-min(1,J_val), min(1,J_val) + 1):
                    G_term = G.subs(J, J_val).subs(LB, LB_val).subs(lam, lam_val).subs(Res, R_symbol)
                    coeff_a = aR[idx_res]
                    term = (
                        coeff_a * G_term * clebsch_gordan(J_val, 1, LB_val, lam_val, -lam_val, 0) *
                        LHA.subs(eta_par, eta).subs(lam1, lambda1).subs(lam2, lambda2) *
                        sp.conjugate(rot.D(1, lam_val, lambda1 - lambda2, 0, thetal, 0)) * 
                        sp.sqrt(sp.Rational(1,4)/pi) * sp.sqrt(sp.Rational((2*J_val + 1),4)/pi) * 
                        sp.conjugate(rot.D(J_val, lam_val, lam_val, phi, theta1, 0)) *
                        sp.conjugate(rot.D(J_val, lam_val, 0, chi, theta2, 0)) *
                        Prop.subs(Res, R_symbol) *
                        BW.subs(L, LB_val).subs(rBW, rB).subs(qR, sp.Symbol('B')) * BW.subs(L, J_val).subs(rBW, r3).subs(qR, R_symbol)
                        )
                    AR += term
                    print(f"\nR = {R_name}, (spin = {spin})")
                    print(f"Processando {R_name} (spin {J_val}) lambda = {lam_val}, LB = {LB_val}")
                    print(f"term = {term}")


# Mostra l'ampiezza totale
print("\n" + "="*50)
if AR == 0:
    print("L'ampiezza AR è nulla (nessuna risonanza selezionata).")
else:
    print("Ampiezza AR totale:")
    print(f"Termini nell'ampiezza AR: {len(AR.as_ordered_terms())}")
print("="*50)

# Mostra l'ampiezza
sp.pprint(AR)



Calcolando contributi per J = 0, LB = 1

R = f0(1500), (spin = 0)
Processando f0(1500) (spin 0) lambda = 0, LB = 1
term = a10*BW(0, r3, f0(1500))*BW(1, rB, B)*G(eta, 0, 0, 1, f0(1500))*LHA(eta, lambda1, lambda2)*R(f0(1500))*conjugate(WignerD(0, 0, 0, chi, theta2, 0))*conjugate(WignerD(0, 0, 0, phi, theta1, 0))*conjugate(WignerD(1, 0, lambda1 - lambda2, 0, thetal, 0))/(4*pi)

R = K0*(1430), (spin = 0)
Processando K0*(1430) (spin 0) lambda = 0, LB = 1
term = a13*BW(0, r3, K0*(1430))*BW(1, rB, B)*G(eta, 0, 0, 1, K0*(1430))*LHA(eta, lambda1, lambda2)*R(K0*(1430))*conjugate(WignerD(0, 0, 0, chi, theta2, 0))*conjugate(WignerD(0, 0, 0, phi, theta1, 0))*conjugate(WignerD(1, 0, lambda1 - lambda2, 0, thetal, 0))/(4*pi)

R = K0*(1950), (spin = 0)
Processando K0*(1950) (spin 0) lambda = 0, LB = 1
term = a16*BW(0, r3, K0*(1950))*BW(1, rB, B)*G(eta, 0, 0, 1, K0*(1950))*LHA(eta, lambda1, lambda2)*R(K0*(1950))*conjugate(WignerD(0, 0, 0, chi, theta2, 0))*conjugate(WignerD(0, 0, 0, phi, theta1, 0))*co

In [7]:
### Cella per costruire l'ampiezza AR1R2 ###
list_resoR1R2 = ['K(1460)', 'K1(1270)', 'K1(1400)', 'K*(1410)', 'K1(1650)', 'K*(1680)', 
              'K2*(1430)', 'K2(1700)', 'K2(1820)', 'K2*(1980)']
# Dizionario delle risonanze che servono per identificare il canale
#(R1, spin1, R2, spin2, tipoR1, L1, L2): coefficient
resonancesR1R2 = {
    ('K(1460)', 0, 'rho(770)', 1, 'K', 1, 1): sp.symbols(f'a17', complex=True), #17
    ('K(1460)', 0, 'K*(892)', 1, 'K', 1, 1): sp.symbols(f'a18', complex=True), #18
    ('K1(1270)', 1, 'rho(770)', 1, 'K', 0, 1): sp.symbols(f'a19', complex=True), #19
    ('K1(1270)', 1, 'omega(782)', 1, 'K', 0, 1): sp.symbols(f'a20', complex=True), #20
    ('K1(1270)', 1, 'K*(892)', 1, 'K', 0, 1): sp.symbols(f'a21', complex=True), #21
    ('K1(1400)', 1, 'K*(892)', 1, 'K', 0, 1): sp.symbols(f'a22', complex=True), #22
    ('K*(1410)', 1, 'K*(892)', 1, 'K*', 1, 1): sp.symbols(f'a23', complex=True), #23
    ('K*(1680)', 1, 'rho(770)', 1, 'K*', 1, 1): sp.symbols(f'a24', complex=True), #24
    ('K*(1680)', 1, 'K*(892)', 1, 'K*', 1, 1): sp.symbols(f'a25', complex=True), #25
    ('K2*(1430)', 2, 'rho(770)', 1, 'K*', 2, 1): sp.symbols(f'a26', complex=True), #26
    ('K2*(1430)', 2, 'omega(782)', 1, 'K*', 2, 1): sp.symbols(f'a27', complex=True), #27
    ('K2*(1430)', 2, 'K*(892)', 1, 'K*', 2, 1): sp.symbols(f'a28', complex=True), #28
    ('K2(1700)', 2, 'K2*(1430)', 2, 'K', 0, 2): sp.symbols(f'a29', complex=True), #29
    ('K2(1700)', 2, 'K*(892)', 1, 'K', 1, 1): sp.symbols(f'a30', complex=True), #30
    ('K2(1820)', 2, 'K2*(1430)', 2, 'K', 0, 2): sp.symbols(f'a31', complex=True), #31
    ('K2(1820)', 2, 'phi(1020)', 1, 'K', 1, 1): sp.symbols(f'a32', complex=True), #32
    ('K2*(1980)', 2, 'phi(1020)', 1, 'K*', 2, 1): sp.symbols(f'a33', complex=True), #33
}



# Dizionario dei coefficienti a(R1,R2) per ogni coppia di risonanze (complessi)
aR1R2 = []
for a in resonancesR1R2.values():
    aR1R2.append(a)
atot.extend(aR1R2)

# --- Costruzione dell'ampiezza AR1R2 ---
AR1R2 = 0
for J1_val in range (0, Jmax + 1):
    for LB_val in range(abs(J1_val-1), abs(J1_val-1) + 1):
        print(f"\n\nCalcolando contributi per J1 = {J1_val}, LB = {LB_val}")
        for key, coeff_a in resonancesR1R2.items():
            R1_name, R1_spin, R2_name, R2_spin, R1_type, L1, L2 = key
            if R1_spin == J1_val:  # Solo se lo spin corrisponde a J1_val
                if R1_type == 'K': auxH = H
                if R1_type == 'K*': auxH = Hbar
                R1_symbol = sp.Symbol(R1_name)
                R2_symbol = sp.Symbol(R2_name)
                J2_val = L2  # Considera solo J2_val = L2
                if R2_spin == J2_val:
                    for L1_val in [L1]:
                        for lam_val in range(-min(1,J1_val), min(1,J1_val) + 1):
                            Hterm = auxH.subs(J, J1_val).subs(LB, LB_val).subs(lam, lam_val).subs(Res, R1_symbol)
                            for lam2_val in range(-min(1,J2_val), min(1,J2_val) + 1):
                                term = (
                                    coeff_a * Hterm * clebsch_gordan(J1_val, 1, LB_val, lam_val, -lam_val, 0) *
                                        LHA.subs(eta_par, eta).subs(lam1, lambda1).subs(lam2, lambda2) *
                                        sp.conjugate(rot.D(1, lam_val, lambda1 - lambda2, 0, thetal, 0)) *
                                        sp.sqrt(sp.Rational((2*J1_val + 1),4)/pi) * sp.sqrt(sp.Rational((2*J2_val + 1),4)/pi) * 
                                        sp.sqrt(sp.Rational((2*L1_val + 1),(2*J1_val + 1))) * 
                                        clebsch_gordan(L1_val, J2_val, J1_val, lam_val-lam2_val, lam2_val, lam_val) *
                                        sp.conjugate(rot.D(J1_val, lam_val, lam2_val, phi, theta1, 0)) *
                                        sp.conjugate(rot.D(J2_val, lam2_val, 0, chi, theta2, 0)) *
                                        Prop.subs(Res, R1_symbol) * Prop.subs(Res, R2_symbol) *
                                        BW.subs(L, LB_val).subs(rBW, rB).subs(qR, sp.Symbol('B')) *
                                        BW.subs(L, L1_val).subs(rBW, r2).subs(qR, R1_symbol) * 
                                        BW.subs(L, J2_val).subs(rBW, r2).subs(qR, R2_symbol)
                                        )
                                AR1R2 += term
                                print(f"\nR1 = {R1_name}, (spin = {R1_spin}), R2 = {R2_name}, (spin = {R2_spin}), lamda = {lam_val}, lam2 = {lam2_val}")
                                print(f"LB = {LB_val}, L1 = {L1}, L2 = {L2}")
                                print(f"term = {term}")

# Mostra l'ampiezza totale
print("\n" + "="*50)
print("Ampiezza AR1R2 totale:")
if AR1R2 == 0:
    print("L'ampiezza AR1R2 è nulla (nessuna risonanza selezionata).")
else:
    print(f"Termini nell'ampiezza AR1R2: {len(AR1R2.as_ordered_terms())}")
print("="*50)



Calcolando contributi per J1 = 0, LB = 1

R1 = K(1460), (spin = 0), R2 = rho(770), (spin = 1), lamda = 0, lam2 = -1
LB = 1, L1 = 1, L2 = 1
term = sqrt(3)*a17*BW(1, r2, K(1460))*BW(1, r2, rho(770))*BW(1, rB, B)*H(eta, 0, 0, 1, K(1460))*LHA(eta, lambda1, lambda2)*R(K(1460))*R(rho(770))*conjugate(WignerD(0, 0, -1, phi, theta1, 0))*conjugate(WignerD(1, -1, 0, chi, theta2, 0))*conjugate(WignerD(1, 0, lambda1 - lambda2, 0, thetal, 0))/(4*pi)

R1 = K(1460), (spin = 0), R2 = rho(770), (spin = 1), lamda = 0, lam2 = 0
LB = 1, L1 = 1, L2 = 1
term = -sqrt(3)*a17*BW(1, r2, K(1460))*BW(1, r2, rho(770))*BW(1, rB, B)*H(eta, 0, 0, 1, K(1460))*LHA(eta, lambda1, lambda2)*R(K(1460))*R(rho(770))*conjugate(WignerD(0, 0, 0, phi, theta1, 0))*conjugate(WignerD(1, 0, 0, chi, theta2, 0))*conjugate(WignerD(1, 0, lambda1 - lambda2, 0, thetal, 0))/(4*pi)

R1 = K(1460), (spin = 0), R2 = rho(770), (spin = 1), lamda = 0, lam2 = 1
LB = 1, L1 = 1, L2 = 1
term = sqrt(3)*a17*BW(1, r2, K(1460))*BW(1, r2, rho(770))*BW(1, 

In [8]:
b = sp.symbols('b', real = True)
ml = sp.symbols('m_l', real = True)

Leptonic_subs = {
    ###Leptonic Helicity Amplitudes MASSLESS
    LHA.subs({eta_par: -1, lam1: sp.Rational(-1,2), lam2: sp.Rational(-1,2)}): 0,
    LHA.subs({eta_par: -1, lam1: sp.Rational(-1,2), lam2: sp.Rational(1,2)}): (2)*sp.sqrt(2)*q,
    LHA.subs({eta_par: -1, lam1: sp.Rational(1,2), lam2: sp.Rational(-1,2)}): 0,
    LHA.subs({eta_par: -1, lam1: sp.Rational(1,2), lam2: sp.Rational(1,2)}): 0,

    LHA.subs({eta_par: 1, lam1: sp.Rational(-1,2), lam2: sp.Rational(-1,2)}): 0,
    LHA.subs({eta_par: 1, lam1: sp.Rational(-1,2), lam2: sp.Rational(1,2)}): 0,
    LHA.subs({eta_par: 1, lam1: sp.Rational(1,2), lam2: sp.Rational(-1,2)}): (2)*sp.sqrt(2)*q,    
    LHA.subs({eta_par: 1, lam1: sp.Rational(1,2), lam2: sp.Rational(1,2)}): 0,
}

In [9]:
# AMPIEZZA TOTALE

print("Verificando la presenza di file salvati...")

# Determina il nome del file
atotsq_file = None

# Cerca i file .pkl nella directory corrente
for file in os.listdir(os.getcwd()):
    if file.startswith("WIGAtotsq_terms") and file.endswith(".pkl"):
        atotsq_file = file

# Controlla se entrambi i file esistono
if atotsq_file and os.path.exists(atotsq_file):
    print("Founded files. Import of saved data.")
    
    try:
        # Carica Atotsq
        with open(atotsq_file, 'rb') as f:
            Atotsqfile = dill.load(f)
            Atotsq = Atotsqfile['Atotsq']
            terms_list = Atotsqfile['terms']
            lenght = Atotsqfile['length']
            print("File loaded successfully.")
        winsound.Beep(1000, 2000)
        
    except Exception as e:
        print(f"❌ Errore durante il caricamento: {e}")
        print("Procedo con il calcolo dall'inizio...")
        atotsq_file = None  # Forza il ricalcolo
        
else:
    print("❌ File non trovato. Procedo con il calcolo dell'ampiezza...")

if not atotsq_file:
    print("Calcolando l'ampiezza totale...")
    A_NR = a_NR * sp.conjugate(rot.D(0, 0, 0, phi, theta1, 0)) * sp.conjugate(rot.D(0, 0, 0, chi, theta2, 0)) * sp.conjugate(rot.D(0, 0, 0, 0, thetal, 0))

    Atot = A_NR + A3 + AR + AR1R2

    totalterms = len(Atot.as_ordered_terms())
    print(f"Numero totale di termini nell'ampiezza Atot: {totalterms}")

    # Sostituisci solo i termini BW con L=0 (non L=1, non L=2)
    W1, W2 = sp.Wild('W1'), sp.Wild('W2')
    Atot = Atot.replace(BWfunc(0, W1, W2), 1)
    
    # Monitora l'uso della memoria
    process = psutil.Process()
    print(f"Memoria utilizzata: {process.memory_info().rss / 1024 / 1024:.1f} MB")
    print(f"Memoria totale: {psutil.virtual_memory().total / 1024 / 1024:.1f} MB")

    # AAcL = (Atot * sp.conjugate(Atot)).xreplace({eta: -1}).xreplace(Leptonic_subs).doit()
    # AAcR = (Atot * sp.conjugate(Atot)).xreplace({eta: 1}).xreplace(Leptonic_subs).doit()

    AAcL = 0
    AAcR = 0
  
    AAcL = (
        (Atot*sp.conjugate(Atot)).subs({eta: -1, lambda1: -sp.Rational(1,2), lambda2: sp.Rational(1,2)}).xreplace(Leptonic_subs)
    )
    print("="*25)
    print(f"Numero termini per eta=-1, lambda1={-sp.Rational(1,2)}, lambda2={sp.Rational(1,2)}: {len(AAcL.as_ordered_terms())}")
    
    AAcR = (
        (Atot*sp.conjugate(Atot)).subs({eta: 1, lambda1: sp.Rational(1,2), lambda2: -sp.Rational(1,2)}).xreplace(Leptonic_subs)
    )
    print("="*25)
    print(f"Numero termini per eta=1, lambda1={sp.Rational(1,2)}, lambda2={-sp.Rational(1,2)}: {len(AAcR.as_ordered_terms())}")
    
    print("\nEspandendo l'espressione finale.")
    Atotsq = sp.expand(AAcL + AAcR)

    # Calcola i termini UNA VOLTA SOLA
    terms_list = Atotsq.as_ordered_terms()
    lenght = len(terms_list)
    print(f"Numero di termini: {lenght}\n")
    print(f"Numero di parametri a_i: {len(atot)}\n")

    for k in range(0,5):
        rnd = randint(0, lenght - 1)
        print(f"Termine casuale #{rnd}:")
        sp.pprint(terms_list[rnd])
        print()

    # Salva ENTRAMBI: l'espressione E i termini già ordinati
    with open(f"WIGAtotsq_terms{lenght}.pkl", 'wb') as f:
        dill.dump({'Atotsq': Atotsq, 'terms': terms_list, 'length': lenght}, f)

    winsound.Beep(1000, 2000)

Verificando la presenza di file salvati...
Founded files. Import of saved data.
File loaded successfully.


In [11]:
from sympy.physics.quantum.spin import WignerD
def resolve_wigner_conjugate(conj_expr):
    """
    Prende in input un oggetto SymPy del tipo conjugate(WignerD(...))
    e lo riscrive come (-1)^(m-k) * WignerD(..., -m, -k, ...).
    
    Args:
        conj_expr: Un'espressione che DEVE essere un oggetto sp.conjugate
                   contenente una WignerD.
    """
    
    # 0. Gestione del caso di potenza di coniugato
    if conj_expr.is_Pow:
        r = sp.S(1)
        p = sp.S(1)
        base, exp = conj_expr.as_base_exp()
        if isinstance(base, sp.conjugate):
            resolved_base, phase = resolve_wigner_conjugate(base)
            r = resolved_base**exp
            p = phase**exp
            return r, p
    
    # 1. Verifica che l'input sia effettivamente un coniugato
    if not isinstance(conj_expr, sp.conjugate):
        # Se l'utente passa una WignerD non coniugata, la restituiamo così com'è
        # oppure potremmo sollevare un errore. Qui restituiamo l'originale.
        return conj_expr
    

    # wigner_try = conj_expr.doit()
    # if sp.im(wigner_try) == 0:
        # return wigner_try, sp.S(1)
    # 2. Estrae l'argomento interno al coniugato (la WignerD "nuda")
    # .args[0] è il modo standard in SymPy per prendere il contenuto di una funzione

    inner_wigner = conj_expr.args[0]
    # 3. Verifica che dentro ci sia davvero una WignerD
    if not isinstance(inner_wigner, WignerD):
        raise TypeError(f"L'oggetto coniugato deve contenere una WignerD, trovato: {type(inner_wigner)}")

    # 4. Estrazione dei parametri originali
    # args[0]=J, args[1]=m, args[2]=k, args[3:]=angoli
    J = inner_wigner.args[0]
    m = inner_wigner.args[1]
    k = inner_wigner.args[2]
    angles = inner_wigner.args[3:] # (alpha, beta, gamma)
    
    # 5. Applicazione della regola: D*_{m,k} = (-1)^(m-k) D_{-m,-k}
    phase = sp.S.NegativeOne**(m - k)
    
    return rot.D(J, -m, -k, *angles), phase

dynamical_coeff = {}
wigner_noconj = []
for term in tqdm(terms_list, desc="Prodotto di Wigner", total = len(terms_list)):
    factor = sp.S(1)
    Wigner1 = sp.S(1)
    Wigner2 = sp.S(1)
    Wigner3 = sp.S(1)

    for single in term.as_ordered_factors():
        str_single = str(single)
        if 'Wigner' in str_single:
            if 'theta1' in str_single:
                Wigner1 *= single
            elif 'theta2' in str_single:
                Wigner2 *= single
            elif 'thetal' in str_single:
                Wigner3 *= single
        else:
            factor *= single  # Mantieni gli altri fattori invariati

    noconj_wiger1 = sp.S(1)
    for Wigner in Wigner1.as_ordered_factors():
        Wigner_str = str(Wigner)
        if "conjugate" in Wigner_str:
            resolved, phase = resolve_wigner_conjugate(Wigner)
            if phase != sp.S(1):
                factor *= phase
        else:
            resolved = Wigner            
        noconj_wiger1 *= resolved
    
    noconj_wiger2 = sp.S(1)
    for Wigner in Wigner2.as_ordered_factors():
        Wigner_str = str(Wigner)
        if "conjugate" in Wigner_str:
            resolved, phase = resolve_wigner_conjugate(Wigner)
            if phase != sp.S(1):
                factor *= phase
        else:
            resolved = Wigner            
        noconj_wiger2 *= resolved
    
    noconj_wiger3 = sp.S(1)
    for Wigner in Wigner3.as_ordered_factors():
        Wigner_str = str(Wigner)
        if "conjugate" in Wigner_str:
            resolved, phase = resolve_wigner_conjugate(Wigner)
            if phase != sp.S(1):
                factor *= phase
        else:
            resolved = Wigner            
        noconj_wiger3 *= resolved

    wigner_final = noconj_wiger1 * noconj_wiger2 * noconj_wiger3
    
    if wigner_final in dynamical_coeff:
        dynamical_coeff[wigner_final] += factor
    else:
        dynamical_coeff[wigner_final] = factor
        wigner_noconj.append(wigner_final)

Prodotto di Wigner:   0%|          | 0/33281 [00:00<?, ?it/s]

In [12]:
from sympy.physics.wigner import clebsch_gordan

def wigner_product_DD(w1, w2):
    """
    Calcola il prodotto di due matrici di Wigner D(j1)*D(j2)
    e lo espande nella serie di Clebsch-Gordan.
    
    Args:
        w1, w2: Oggetti sympy di tipo WignerD
        
    Returns:
        Somma di termini: Coeff * WignerD(J, M, K, angoli)
    """
    
    # 1. Verifica input
    if not isinstance(w1, WignerD) or not isinstance(w2, WignerD):
        raise TypeError("Entrambi gli argomenti devono essere oggetti WignerD")
        
    # 2. Estrazione Parametri
    j1, m1, k1 = w1.args[0], w1.args[1], w1.args[2]
    angles1 = w1.args[3:]
    
    j2, m2, k2 = w2.args[0], w2.args[1], w2.args[2]
    angles2 = w2.args[3:]
    
    # 3. Verifica che gli angoli siano gli stessi
    if angles1 != angles2:
        return w1 * w2 # Ritorna il prodotto non espanso se gli angoli differiscono

    # 4. Calcolo Numeri Quantici Totali
    M = m1 + m2
    K = k1 + k2
    
    # 5. Limiti della somma su J
    # Nota: richiede che J siano valori numerici o simboli risolvibili
    try:
        J_min = int(abs(j1 - j2))
        J_max = int(j1 + j2)
    except TypeError:
        print("Attenzione: J simbolici, impossibile espandere la serie.")
        return w1 * w2

    expansion = 0
    
    # 6. Loop sulla serie di Clebsch-Gordan
    for J_curr in range(J_min, J_max + 1):
        J = sp.S(J_curr)
        
        # Coefficienti CG: <j1 m1 j2 m2 | J M>
        # Sintassi SymPy: clebsch_gordan(j1, j2, j3, m1, m2, m3)
        cg_M = clebsch_gordan(j1, j2, J, m1, m2, M)
        
        if cg_M != 0:
            # Coefficienti CG: <j1 k1 j2 k2 | J K>
            cg_K = clebsch_gordan(j1, j2, J, k1, k2, K)
            
            if cg_K != 0:
                # Costruisce il termine
                term = cg_M * cg_K * WignerD(J, M, K, *angles1)
                expansion += term
                
    return expansion

In [13]:
angular_basis_file = None
angular_trig_file = None

# Cerca i file .pkl nella directory corrente
for file in os.listdir(os.getcwd()):
    if file.startswith("angular_basis") and file.endswith(".pkl"):
        angular_basis_file = file
    if file.startswith("angular_trig_coefficients") and file.endswith(".pkl"):
        angular_trig_file = file
if angular_basis_file and angular_trig_file:
    print("Founded files. Import of saved data.")
    
    try:
        # Carica angular_basis
        with open(angular_basis_file, 'rb') as f:
            angular_basis = dill.load(f)
        # Carica angular_trig_coefficients
        with open(angular_trig_file, 'rb') as f:
            angular_trig_coefficients = dill.load(f)
        print("Files loaded successfully.")
        winsound.Beep(1000, 2000)
        
    except Exception as e:
        print(f"❌ Errore durante il caricamento: {e}")
        print("Procedo con il calcolo dall'inizio...")
        angular_basis_file = None  # Forza il ricalcolo
        angular_trig_file = None

if not angular_basis_file or not angular_trig_file:

    angular_basis = []
    angular_trig_coefficients = {}

    for wigner_term in tqdm(wigner_noconj, desc="Prodotto di Wigner D", total=len(wigner_noconj)):

        # Controllo: se wigner_term è 1, crea WignerD con indici di spin a zero
        if wigner_term == sp.S(1):
            wigner_group1 = rot.D(0, 0, 0, phi, theta1, 0)
            wigner_group2 = rot.D(0, 0, 0, chi, theta2, 0)
            wigner_group3 = rot.D(0, 0, 0, 0, thetal, 0)
        else:
            wigner_group1 = sp.S(1)
            wigner_group2 = sp.S(1)
            wigner_group3 = sp.S(1)

            for factor in wigner_term.as_ordered_factors():
                str_factor = str(factor)
                if 'theta1' in str_factor:
                    wigner_group1 *= factor
                elif 'theta2' in str_factor:
                    wigner_group2 *= factor
                elif 'thetal' in str_factor:
                    wigner_group3 *= factor
        
        # ========== GRUPPO 1 (theta1, phi) ==========
        # Inizializza a identità
        wigner_group1_expanded = wigner_group1
        
        # Caso 1: Potenza (es. WignerD^2)
        if wigner_group1.is_Pow:
            base, exp = wigner_group1.as_base_exp()
            if exp == 2:
                wigner_group1_expanded = wigner_product_DD(base, base)
                
        # Caso 2: Prodotto di fattori (es. WignerD * WignerD)
        elif wigner_group1.is_Mul:
            wigner_group1_factor_list = wigner_group1.as_ordered_factors()
            if len(wigner_group1_factor_list) > 1:
                A = wigner_group1_factor_list[0]
                B = wigner_group1_factor_list[1]
                wigner_group1_expanded = wigner_product_DD(A, B)
        # Caso 3: Singolo WignerD o 1 → lascia com'è

        # ========== GRUPPO 2 (theta2, chi) ==========
        wigner_group2_expanded = wigner_group2
        
        if wigner_group2.is_Pow:
            base, exp = wigner_group2.as_base_exp()
            if exp == 2:
                wigner_group2_expanded = wigner_product_DD(base, base)
        elif wigner_group2.is_Mul:
            wigner_group2_factor_list = wigner_group2.as_ordered_factors()
            if len(wigner_group2_factor_list) > 1:
                A = wigner_group2_factor_list[0]
                B = wigner_group2_factor_list[1]
                wigner_group2_expanded = wigner_product_DD(A, B)

        # ========== GRUPPO 3 (thetal) ==========
        wigner_group3_expanded = wigner_group3
        
        if wigner_group3.is_Pow:
            base, exp = wigner_group3.as_base_exp()
            if exp == 2:
                wigner_group3_expanded = wigner_product_DD(base, base)
        elif wigner_group3.is_Mul:
            wigner_group3_factor_list = wigner_group3.as_ordered_factors()
            if len(wigner_group3_factor_list) > 1:
                A = wigner_group3_factor_list[0]
                B = wigner_group3_factor_list[1]
                wigner_group3_expanded = wigner_product_DD(A, B)

        # ========== ESPANSIONE FINALE ==========
        wigner_final_expanded = (wigner_group1_expanded * wigner_group2_expanded * wigner_group3_expanded).expand()
        
        # Accumula i coefficienti
        for DDD in wigner_final_expanded.as_ordered_terms():

            if DDD == sp.S(1) or DDD == 1:
                DDD_elementbase = rot.D(0, 0, 0, phi, theta1, 0) * rot.D(0, 0, 0, chi, theta2, 0) * rot.D(0, 0, 0, 0, thetal, 0)
                coeff_total = sp.expand(dynamical_coeff[wigner_term])
                if DDD_elementbase in angular_trig_coefficients:
                    angular_trig_coefficients[DDD_elementbase] += coeff_total
                else:
                    angular_trig_coefficients[DDD_elementbase] = coeff_total
                    angular_basis.append(DDD_elementbase)
                continue

            DDD_elementbase = sp.S(1)
            num_coeff = sp.S(1)
            
            for Dfactor in DDD.as_ordered_factors():
                Dstr = str(Dfactor)
                if 'Wigner' in Dstr:
                    DDD_elementbase *= Dfactor
                else:
                    num_coeff *= Dfactor
            
            coeff_total = sp.expand(num_coeff * dynamical_coeff[wigner_term])
            
            if DDD_elementbase == sp.S(1) or DDD_elementbase == 1:
                DDD_elementbase = rot.D(0, 0, 0, phi, theta1, 0) * rot.D(0, 0, 0, chi, theta2, 0) * rot.D(0, 0, 0, 0, thetal, 0)

            if DDD_elementbase in angular_trig_coefficients:
                angular_trig_coefficients[DDD_elementbase] += coeff_total
            else:
                angular_trig_coefficients[DDD_elementbase] = coeff_total
                angular_basis.append(DDD_elementbase)
                print(DDD_elementbase)
    # Ordina angular_basis per lunghezza della stringa (prima i più semplici)
    angular_basis = sorted(angular_basis, key=lambda x: (len(str(x)), str(x)))

    # Riorganizza angular_trig_coefficients per seguire l'ordine di angular_basis
    angular_trig_coefficients = {key: angular_trig_coefficients[key] for key in angular_basis}
    angular_trig_coefficients = angular_trig_coefficients

    with open("angular_trig_coefficients.pkl", 'wb') as f:
            dill.dump(angular_trig_coefficients, f)
            print("Dati salvati in 'angular_trig_coefficients.pkl'")

    with open("angular_basis.pkl", 'wb') as f:
        dill.dump(angular_basis, f)
        print("Chiavi salvate in 'angular_basis.pkl'")

Founded files. Import of saved data.
Files loaded successfully.


In [14]:
sorted_files = None
trig_coefficients_file = None
# Cerca i file .pkl nella directory corrente
for file in os.listdir(os.getcwd()):
    if file.startswith("sorted_keys") and file.endswith(".pkl"):
        sorted_files = file
    if file.startswith("trig_coefficients") and file.endswith(".pkl"):
        trig_coefficients_file = file

if sorted_files and trig_coefficients_file and os.path.exists(sorted_files) and os.path.exists(trig_coefficients_file):
    print("Founded files. Import of saved data.")
    
    try:
        # Carica sorted_keys
        with open(sorted_files, 'rb') as f:
            sorted_keys = dill.load(f)
        # Carica trig_coefficients
        with open(trig_coefficients_file, 'rb') as f:
            trig_coefficients = dill.load(f)
        print("Files loaded successfully.")
        winsound.Beep(1000, 2000)
        
    except Exception as e:
        print(f"❌ Errore durante il caricamento: {e}")
        print("Procedo con il calcolo dall'inizio...")
        sorted_files = None  # Forza il ricalcolo
        trig_coefficients_file = None

if not sorted_files or not trig_coefficients_file:
    sorted_keys = []
    trig_coefficients = {}

    used = set()
    for DDD in tqdm(angular_basis, desc="Coefficienti finali", total=len(angular_basis)):
        if DDD in used:
            continue
        to_find = sp.S(1)
        phase_prod = 1
        
        A = angular_trig_coefficients[DDD]

        for D in DDD.as_ordered_factors():
            J = D.args[0]
            M = D.args[1]
            K = D.args[2]
            angles = D.args[3:]

            to_find *= rot.D(J, -M, -K, *angles)
            phase_prod *= sp.S.NegativeOne**(M - K)
        
        # In questo caso sono già reali
        if DDD == to_find:
            sorted_keys.append(DDD)
            trig_coefficients[DDD] = angular_trig_coefficients[DDD]
            
            used.add(DDD)

        # Caso coniugati diversi
        if not to_find in used and DDD != to_find:
            if not to_find in angular_basis:
                print("Non trovato il coniugato!")
                continue

            
            elif to_find in angular_basis:
                used.add(DDD)
                used.add(to_find)
                print("\n========================")
                
                A_star = angular_trig_coefficients[to_find]

                if sp.conjugate(A) == A_star:
                    eta_A = 1
                elif sp.conjugate(A) == -A_star:
                    eta_A = -1
                else:
                    raise ValueError("Parità dinamica non definita")

                # Parità del blocco DA
                block_phase = phase_prod * eta_A

                # Coefficiente finale (la combinazione giusta è già fissata da block_phase)
                new_coeff = A + block_phase * A_star

                sp.pprint(DDD)
                sp.pprint(to_find)
                print(f"Phase: {phase_prod}")
                new_key = sp.S(1)
                
                D_block = DDD

                if block_phase == 1:
                    new_key = sp.re(D_block)
                else:
                    new_key = sp.im(D_block)
                
                sp.pprint(new_key)
                sorted_keys.append(new_key)
                trig_coefficients[new_key] = sp.expand(new_coeff)
                if phase_prod == -1:
                    print("cambio phase:")
                    sp.pprint(trig_coefficients[new_key])


    with open("trig_coefficients.pkl", 'wb') as f:
        dill.dump(trig_coefficients, f)
        print("Dati salvati in 'trig_coefficients.pkl'")
    with open("sorted_keys.pkl", 'wb') as f:
        dill.dump(sorted_keys, f)
        print("Chiavi salvate in 'sorted_keys.pkl'")

Founded files. Import of saved data.
Files loaded successfully.


In [15]:
print(len(sorted_keys))

295


In [16]:
symmetry_file = None

# Cerca file symmetry*.pkl
for file in os.listdir(os.getcwd()):
    if file.startswith("symmetry") and file.endswith(".pkl"):
        symmetry_file = file

if symmetry_file and os.path.exists(symmetry_file):
    print("Founded files. Import of saved data.")
    try:
        with open(symmetry_file, 'rb') as f:
            symmetry = dill.load(f)
            print("File loaded successfully.")

            print(f"symmetry of terms:")
            for k in range(len(sorted_keys)):
                print(f"f_{k+1} = {sorted_keys[k]}. Simmetria = {symmetry.get(sorted_keys[k])}")


            winsound.Beep(1000, 500)
    except Exception as e:
        print(f"❌ Errore durante il caricamento: {e}")
        print("Procedo con il calcolo dall'inizio...")
        symmetry_file = None

else:
    symmetry = {}

    for idx_key, key in tqdm(
        enumerate(sorted_keys),
        desc="Trovando la Simmetria",
        total=len(sorted_keys)
    ):

        if key == "COSTANTE":
            symmetry[key] = 1
            print(f"Key #{idx_key+1}: {key} --> symmetry = 1 (COSTANTE)")
            continue

        
        listed_coeff = trig_coefficients[key].as_ordered_terms()
        first_term = listed_coeff[0]
        # Crea wildcard per le risonanze
        w_res = sp.Wild('w_res')
        
        excluded_names = {'H', 'G', 'Hbar'}

        # --- 1. Calcolo il Pure Term di Riferimento ---
        ref_pure = 1
        for factor in first_term.as_ordered_factors():
            is_excluded = False
            atoms_in_factor = factor.atoms(sp.Function, sp.Symbol)
            for atom in atoms_in_factor:
                name = atom.name if hasattr(atom, 'name') else atom.func.__name__
                if name in excluded_names:
                    is_excluded = True
                    break
            if not is_excluded:
                ref_pure *= factor

        # --- 2. Cerco il partner per determinare la Simmetria ---
        # Default: se non trovo nulla, decidi tu cosa mettere (es. None o 0)
        detected_symmetry = None 
        
        # Itero sugli altri termini (salto il primo che è il riferimento)
        for term in listed_coeff[1:]:
            current_pure = 1
            for factor in term.as_ordered_factors():
                is_excluded = False
                atoms_in_factor = factor.atoms(sp.Function, sp.Symbol)
                for atom in atoms_in_factor:
                    name = atom.name if hasattr(atom, 'name') else atom.func.__name__
                    if name in excluded_names:
                        is_excluded = True
                        break
                if not is_excluded:
                    current_pure *= factor
            
            # --- CONFRONTO DEI SEGNI ---
            
            # Caso A: Segno OPPOSTO (Antisimmetrico) -> Es. Simmetria -1
            if current_pure == -ref_pure:
                detected_symmetry = -1 
                break # Trovato, possiamo uscire
            
            # Caso B: Segno UGUALE (Simmetrico) -> Es. Simmetria +1
            elif current_pure == ref_pure:
                detected_symmetry = 1
                break

        # Se non ha trovato partner (es. termine unico), gestisci il caso:
        if detected_symmetry is None:
            print("  -> Nessun termine partner trovato (termine singolo).")
            detected_symmetry = 0

        symmetry[key] = detected_symmetry
        print(f"Key #{idx_key+1}: {key} --> symmetry = {detected_symmetry}")


    # Salvataggio finale
    with open("symmetry.pkl", 'wb') as f:
        dill.dump(symmetry, f)


Founded files. Import of saved data.
File loaded successfully.
symmetry of terms:
f_1 = WignerD(0, 0, 0, 0, thetal, 0)*WignerD(0, 0, 0, chi, theta2, 0)*WignerD(0, 0, 0, phi, theta1, 0). Simmetria = 1
f_2 = WignerD(0, 0, 0, 0, thetal, 0)*WignerD(0, 0, 0, chi, theta2, 0)*WignerD(1, 0, 0, phi, theta1, 0). Simmetria = 1
f_3 = WignerD(0, 0, 0, 0, thetal, 0)*WignerD(0, 0, 0, chi, theta2, 0)*WignerD(2, 0, 0, phi, theta1, 0). Simmetria = 1
f_4 = WignerD(0, 0, 0, 0, thetal, 0)*WignerD(0, 0, 0, chi, theta2, 0)*WignerD(3, 0, 0, phi, theta1, 0). Simmetria = 1
f_5 = WignerD(0, 0, 0, 0, thetal, 0)*WignerD(0, 0, 0, chi, theta2, 0)*WignerD(4, 0, 0, phi, theta1, 0). Simmetria = 1
f_6 = WignerD(0, 0, 0, 0, thetal, 0)*WignerD(0, 0, 0, phi, theta1, 0)*WignerD(1, 0, 0, chi, theta2, 0). Simmetria = 1
f_7 = WignerD(0, 0, 0, 0, thetal, 0)*WignerD(0, 0, 0, phi, theta1, 0)*WignerD(2, 0, 0, chi, theta2, 0). Simmetria = -1
f_8 = WignerD(0, 0, 0, 0, thetal, 0)*WignerD(0, 0, 0, phi, theta1, 0)*WignerD(3, 0, 0, chi,

In [17]:
LEFT_trig_coefficients_file = None

# Cerca file LEFT_trig_coefficients*.pkl
for file in os.listdir(os.getcwd()):
    if file.startswith("LEFT_trig_coefficients") and file.endswith(".pkl"):
        LEFT_trig_coefficients_file = file

if LEFT_trig_coefficients_file and os.path.exists(LEFT_trig_coefficients_file):
    print("Founded files. Import of saved data.")
    try:
        with open(LEFT_trig_coefficients_file, 'rb') as f:  # FIX: era symmetry_file
            LEFT_trig_coefficients = dill.load(f)
            print("File LEFT_trig_coefficients caricato correttamente.")
            winsound.Beep(1000, 500)
    except Exception as e:
        print(f"❌ Errore durante il caricamento: {e}")
        print("Procedo con il calcolo dall'inizio...")
        LEFT_trig_coefficients_file = None

if not LEFT_trig_coefficients_file:
    # Filtra i termini con eta=-1 da tutti i coefficienti
    LEFT_trig_coefficients = {}
    print("Filtrando i termini con eta=-1 (Left-handed)...")

    for key in tqdm(sorted_keys, desc="Filtraggio termini L", total=len(sorted_keys)):
        if key == "COSTANTE":
            LEFT_trig_coefficients[key] = sp.Rational(1, 2) * trig_coefficients[key]
            continue
        
        coeff = trig_coefficients[key]
        
        if coeff == 0:
            LEFT_trig_coefficients[key] = 0
            continue
        
        terms = coeff.as_ordered_terms()
        filtered_terms = []
        
        for term in terms:
            has_L = False
            
            for factor in term.as_ordered_factors():
                # Funzione helper per estrarre eta da H, Hbar, G
                def check_eta(func_obj):
                    if hasattr(func_obj, 'func'):
                        func_name = func_obj.func.__name__ if hasattr(func_obj.func, '__name__') else str(func_obj.func)
                        if func_name in ['H', 'Hbar', 'G']:
                            if len(func_obj.args) > 0:
                                return func_obj.args[0]  # Ritorna il valore di eta
                    return None
                
                # Caso 1: Funzione diretta H(-1, ...), Hbar(-1, ...), G(-1, ...)
                eta_val = check_eta(factor)
                if eta_val == -1:
                    has_L = True
                    break
                
                # Caso 2: Coniugato conjugate(H(-1, ...))
                if hasattr(factor, 'func') and factor.func.__name__ == 'conjugate':
                    # Estrai la funzione interna dal coniugato
                    inner_func = factor.args[0]  # Questo è H(-1, ...) dentro conjugate(...)
                    eta_val = check_eta(inner_func)
                    if eta_val == -1:
                        has_L = True
                        break
            
            if has_L:
                filtered_terms.append(term)
        
        if filtered_terms:
            LEFT_trig_coefficients[key] = sum(filtered_terms)
        else:
            LEFT_trig_coefficients[key] = sp.S(0)

    # Statistiche
    print(f"\n" + "="*70)
    print("RISULTATI FILTRAGGIO:")
    print("="*70)

    original = sp.S(0)
    left = sp.S(0)
    for k in range(len(sorted_keys)):
        original += len(trig_coefficients[sorted_keys[k]].as_ordered_terms())
        left += len(LEFT_trig_coefficients[sorted_keys[k]].as_ordered_terms())
    print(f"- Termini totali originali: {original}")
    print(f"- Termini totali Left-handed: {left}")

    # Salva
    with open("LEFT_trig_coefficients.pkl", 'wb') as f:
        dill.dump(LEFT_trig_coefficients, f)
        print(f"\nDati salvati in 'LEFT_trig_coefficients.pkl'")

    winsound.Beep(1000, 500)

Founded files. Import of saved data.
File LEFT_trig_coefficients caricato correttamente.


In [18]:
print("Semplificando i coefficienti: z × z* → |z|²...")

SIMPLIFIED_LEFT_trig_coefficients_file = None

# Cerca file SIMPLIFIED_LEFT_trig_coefficients*.pkl
for file in os.listdir(os.getcwd()):
    if file.startswith("SIMPLIFIED_LEFT_trig_coefficients") and file.endswith(".pkl"):
        SIMPLIFIED_LEFT_trig_coefficients_file = file

if SIMPLIFIED_LEFT_trig_coefficients_file and os.path.exists(SIMPLIFIED_LEFT_trig_coefficients_file):
    print("Founded files. Import of saved data.")
    try:
        with open(SIMPLIFIED_LEFT_trig_coefficients_file, 'rb') as f:
            SIMPLIFIED_LEFT_trig_coefficients = dill.load(f)
            print("File SIMPLIFIED_LEFT_trig_coefficients caricato correttamente.")
            winsound.Beep(1000, 500)
    except Exception as e:
        print(f"❌ Errore durante il caricamento: {e}")
        print("Procedo con il calcolo dall'inizio...")
        SIMPLIFIED_LEFT_trig_coefficients_file = None

if not SIMPLIFIED_LEFT_trig_coefficients_file:
    SIMPLIFIED_LEFT_trig_coefficients = {}

    for key in tqdm(sorted_keys, desc="Semplificazione algebrica", total=len(sorted_keys)):
        
        coeff = LEFT_trig_coefficients[key]
        
        if coeff == 0:
            SIMPLIFIED_LEFT_trig_coefficients[key] = 0
            continue
        
        # Scomponi in termini
        terms = coeff.as_ordered_terms()
        simplified_terms = []
        
        for term in terms:
            # Scomponi il termine in fattori
            factors = list(term.as_ordered_factors())
            
            # Cerca coppie z e conjugate(z)
            new_factors = []
            used_indices = set()
            
            for i, factor_i in enumerate(factors):
                if i in used_indices:
                    continue
                
                # Cerca se esiste il coniugato di questo fattore
                found_conjugate = False
                
                for j, factor_j in enumerate(factors):
                    if i == j or j in used_indices:
                        continue
                    
                    # Caso 1: factor_j è conjugate(factor_i)
                    if (hasattr(factor_j, 'func') and 
                        factor_j.func.__name__ == 'conjugate' and 
                        len(factor_j.args) > 0 and 
                        factor_j.args[0] == factor_i):
                        
                        # Sostituisci con |factor_i|²
                        new_factors.append(sp.Abs(factor_i)**2)
                        used_indices.add(i)
                        used_indices.add(j)
                        found_conjugate = True
                        break
                    
                    # Caso 2: factor_i è conjugate(factor_j)
                    if (hasattr(factor_i, 'func') and 
                        factor_i.func.__name__ == 'conjugate' and 
                        len(factor_i.args) > 0 and 
                        factor_i.args[0] == factor_j):
                        
                        # Sostituisci con |factor_j|²
                        new_factors.append(sp.Abs(factor_j)**2)
                        used_indices.add(i)
                        used_indices.add(j)
                        found_conjugate = True
                        break
                
                if not found_conjugate:
                    new_factors.append(factor_i)
                    used_indices.add(i)
            
            # Ricostruisci il termine
            if new_factors:
                simplified_term = sp.Mul(*new_factors)
                simplified_terms.append(simplified_term)
        
        # Ricostruisci il coefficiente
        if simplified_terms:
            SIMPLIFIED_LEFT_trig_coefficients[key] = sum(simplified_terms)
            
        else:
            SIMPLIFIED_LEFT_trig_coefficients[key] = 0

    # Salva
    with open("SIMPLIFIED_LEFT_trig_coefficients.pkl", 'wb') as f:
        dill.dump(SIMPLIFIED_LEFT_trig_coefficients, f)
        print(f"\n{'='*70}")
        print(f"Dati salvati in 'SIMPLIFIED_LEFT_trig_coefficients.pkl'")
    winsound.Beep(1000, 500)

Semplificando i coefficienti: z × z* → |z|²...
Founded files. Import of saved data.
File SIMPLIFIED_LEFT_trig_coefficients caricato correttamente.


In [19]:
print("Semplificando termini coniugati: z±conj(z) → 2Re(z) o 2iIm(z)...")

FINAL_trig_coefficients_file = None
for file in os.listdir(os.getcwd()):
    if file.startswith("FINAL_trig_coefficients") and file.endswith(".pkl"):
        FINAL_trig_coefficients_file = file

if FINAL_trig_coefficients_file and os.path.exists(FINAL_trig_coefficients_file):
    print("Founded files. Import of saved data.")
    try:
        with open(FINAL_trig_coefficients_file, 'rb') as f:
            FINAL_trig_coefficients = dill.load(f)
            print("File FINAL_trig_coefficients caricato correttamente.")
            winsound.Beep(1000, 500)
    except Exception as e:
        print(f"❌ Errore durante il caricamento: {e}")
        print("Procedo con il calcolo dall'inizio...")
        FINAL_trig_coefficients_file = None

if not FINAL_trig_coefficients_file:

    FINAL_trig_coefficients = {}

    for key in tqdm(sorted_keys, desc="Semplificazione z±conj(z)", total=len(sorted_keys)):
        
        coeff = SIMPLIFIED_LEFT_trig_coefficients[key]
        
        if coeff == 0:
            FINAL_trig_coefficients[key] = 0
            continue
        
        # Scomponi in termini
        terms = list(coeff.as_ordered_terms())
        
        new_terms = []
        used_indices = set()
        
        for i, term_i in enumerate(terms):
            if i in used_indices:
                continue
            
            found_conjugate = False
            
            # Cerca il coniugato tra gli altri termini
            for j, term_j in enumerate(terms):
                if i == j or j in used_indices:
                    continue
                
                # Controlla se term_j == conjugate(term_i) -> SOMMA
                if term_j == sp.conjugate(term_i):
                    used_indices.add(i)
                    used_indices.add(j)
                    found_conjugate = True
                    # z + conj(z) = 2*Re(z)
                    new_terms.append(2 * sp.re(term_i))
                    break
                
                # Controlla se term_j == -conjugate(term_i) -> DIFFERENZA
                elif term_j == -sp.conjugate(term_i):
                    used_indices.add(i)
                    used_indices.add(j)
                    found_conjugate = True
                    # z - conj(z) = 2i*Im(z), ma siccome z contiene già I,
                    # scriviamo direttamente: (z - conj(z))/i = 2*Im(z) reale
                    # Oppure lasciamo: i*(z - conj(z))/2i = (z - conj(z))/2
                    # Meglio: z - conj(z) è puramente immaginario, quindi:
                    new_terms.append(term_i - sp.conjugate(term_i))
                    break
            
            if not found_conjugate:
                # Nessun coniugato trovato, mantieni il termine
                new_terms.append(term_i)
                used_indices.add(i)
        
        # Ricostruisci il coefficiente
        if new_terms:
            FINAL_trig_coefficients[key] = sum(new_terms)

        else:
            FINAL_trig_coefficients[key] = 0

    # Salva
    with open("FINAL_trig_coefficients.pkl", 'wb') as f:
        dill.dump(FINAL_trig_coefficients, f)
        print(f"\nDati salvati in 'FINAL_trig_coefficients.pkl'")

    winsound.Beep(1500, 500)

Semplificando termini coniugati: z±conj(z) → 2Re(z) o 2iIm(z)...
Founded files. Import of saved data.
File FINAL_trig_coefficients caricato correttamente.


In [20]:
count = 0
for x in tqdm(sorted_keys, total = len(sorted_keys)):
    if sp.im(x.doit()) != 0:
        count += 1
print(f"{count}/{len(sorted_keys)} hanno parte immaginaria non nulla.")

  0%|          | 0/295 [00:00<?, ?it/s]

0/295 hanno parte immaginaria non nulla.


In [ ]:
count = 0
for x in tqdm(sorted_keys, total = len(sorted_keys)):
    coeff = FINAL_trig_coefficients[x]
    if sp.im(coeff) != 0:
        final_term = sp.S(0)
        used = set()
        term_lists = coeff.as_ordered_terms()
        for term in term_lists:
            aux = sp.conjugate(term)
            if aux not in used:
                used.add(term)
                if aux == term:
                    final_term += sp.re(term)
                    used.add(aux)
                else:
                    final_term += sp.im(term)
                    used.add(-aux)
        FINAL_trig_coefficients[x] = final_term
        count += 1

moments = FINAL_trig_coefficients

with open("Moments.pkl", 'wb') as f:
        dill.dump(moments, f)
        print(f"\nDati salvati in 'Moments.pkl'")

  0%|          | 0/295 [00:00<?, ?it/s]


Dati salvati in 'Moments.pkl'


In [22]:
import numpy as np

class AngularIntegrator:
    def __init__(self):
        # Variabili mute per l'integrazione (interne)
        self._t1_int, self._t2_int, self._tl_int = sp.symbols('t1_int t2_int tl_int', real=True, positive=True)
        self._phi_int, self._chi_int = sp.symbols('phi_int chi_int', real=True)
        
        # Mappa: Nome Variabile (str) -> Variabile Muta (Symbol)
        self.target_vars = {
            'theta1': self._t1_int,
            'theta2': self._t2_int,
            'thetal': self._tl_int,
            'phi':    self._phi_int,
            'chi':    self._chi_int
        }

        # Fattore di normalizzazione

    def scalarprod(self, func1, func2):
        """
        Calcola <f1|f2>. Sostituisce le variabili basandosi sul NOME.
        """

        j_prod = 1
        
        # all_factors = func1.as_ordered_factors() + func2.as_ordered_factors()
        # for D in all_factors:
        #     if hasattr(D, 'func') and D.func.__name__ == 'conjugate':
        #         D = D.args[0]
        #     if isinstance(D, WignerD):
        #         j_curr = D.args[0]
        #         j_prod *= (2 * j_curr + 1)
        
        norm_factor = j_prod / ( (2*sp.pi)**2 * 2**3 )
        
        # 1. Crea il prodotto
        product = func1 * sp.conjugate(func2)
        # 2. Sostituzione ROBUSTA (Per Nome)
        # Cerca tra i simboli liberi dell'espressione. Se il nome corrisponde, sostituisci.
        subs_map = {}
        for sym in product.free_symbols:
            if sym.name in self.target_vars:
                subs_map[sym] = self.target_vars[sym.name]
        
        # Applica la sostituzione: ora le tue theta diventano t_int
        product_int = product.subs(subs_map)
        
        # 3. Jacobiano (usando le variabili mute)
        jacobian = sp.sin(self._t1_int) * sp.sin(self._t2_int) * sp.sin(self._tl_int)
        integrand = product_int * jacobian
        
        # 4. Integrazione
        res = integrand
        # Angoli azimutali [0, 2pi]
        res = sp.integrate(res, (self._phi_int, 0, 2*sp.pi))
        res = sp.integrate(res, (self._chi_int, 0, 2*sp.pi))
        # Angoli polari [0, pi]
        res = sp.integrate(res, (self._t1_int, 0, sp.pi))
        res = sp.integrate(res, (self._t2_int, 0, sp.pi))
        res = sp.integrate(res, (self._tl_int, 0, sp.pi))
        
        return sp.simplify(res * norm_factor)

integrator = AngularIntegrator()

In [25]:
seeds = [16636, 123, 2024]
np.random.seed(seeds[2])
rdn = np.random.rand()
f = sorted_keys[int(rdn * len(sorted_keys))]
sp.pprint(f)
np.random.seed(seeds[1])
rdn = np.random.rand()
g = sorted_keys[int(rdn * len(sorted_keys))]
sp.pprint(g)
ris = integrator.scalarprod(f, g)
print(abs)
if abs != 0:
    sp.pprint(f"f = {f}")
    sp.pprint(f"g = {g}")
    print(f"<f|g> = {ris}")
            

  ⎛ 1            2            4          ⎞
re⎜D   (χ,θ₂,0)⋅d   (thetal)⋅D   (φ,θ₁,0)⎟
  ⎝ 1,0          0,0          0,1        ⎠
 2            4            4          
d   (thetal)⋅D   (χ,θ₂,0)⋅D   (φ,θ₁,0)
 0,0          0,0          0,0        
0


In [26]:
L = len(sorted_keys)
p = 0
m = 0
for x in sorted_keys:
    if symmetry[x] == 1:
        p += 1
    if symmetry[x] == -1:
        m += 1
print(f"{p}/{L} funzioni di base con simmetria +(L -> R)")
print(f"{m}/{L} funzioni di base con simmetria -(L -> R)")

170/295 funzioni di base con simmetria +(L -> R)
125/295 funzioni di base con simmetria -(L -> R)
